# 06 — Exploratory: fouls or cards?

> **EXPLORATORY — NOT PRE-REGISTERED.** These analyses were added after seeing the main results (notebooks 03–05).
> They describe a possible mechanism; they are not tests of the pre-registered hypotheses and must be reported as exploratory.

**Question.** The no-crowd drop in the away-team card excess (−0.225 per match) could come from two places:
1. **fouls:** without a crowd, the foul balance shifts against the home team, so with the same cards-per-foul rate the card balance shifts too;
2. **cards per foul:** for the same foul, the referee's card decision changes.

**Limitation that no analysis here removes:** foul counts mix referee decisions (whistle or not) with player behaviour (how many fouls are committed). A change in fouls cannot be attributed to the referee alone.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import statsmodels.formula.api as smf
from wildboottest.wildboottest import wildboottest

PROCESSED_DIR = Path("../data/processed")
pd.set_option("display.width", 200)

matches = pd.read_csv(PROCESSED_DIR / "matches_elo.csv", parse_dates=["Date"])
analysis = matches[matches["season_start"] >= 2017].copy()

MAIN_RIGHT_SIDE = "ghost + partial + elo_diff + var + trend + C(Referee)"


def fit_ghost_model(data, outcome, right_side=MAIN_RIGHT_SIDE, param="ghost"):
    """Same estimator as notebook 04: OLS, SEs clustered by referee, wild cluster bootstrap p-value."""
    data = data.dropna(subset=[outcome]).reset_index(drop=True)
    referee_ids = pd.factorize(data["Referee"])[0]
    model = smf.ols(f"{outcome} ~ {right_side}", data=data)
    fitted = model.fit(cov_type="cluster", cov_kwds={"groups": referee_ids})
    boot = wildboottest(model, param=param, cluster=referee_ids, B=9999,
                        weights_type="rademacher", impose_null=True, seed=2026, show=False)
    return {
        "outcome": outcome,
        "coef": fitted.params[param],
        "ci_low": fitted.conf_int().loc[param, 0],
        "ci_high": fitted.conf_int().loc[param, 1],
        "p_bootstrap": float(boot["p-value"].iloc[0]),
        "mde": 2.8 * fitted.bse[param],
        "full_crowd_mean": data.loc[data["crowd"] == "full", outcome].mean(),
    }

## 1. Foul difference with the main model

Same controls as the main model (ELO, VAR, trend, referee fixed effects).

In [2]:
fouls_result = pd.DataFrame([fit_ghost_model(analysis, "foul_diff")])
fouls_result.round(3)

,outcome,coef,ci_low,ci_high,p_bootstrap,mde,full_crowd_mean
0,foul_diff,-1.014,-1.682,-0.345,0.008,0.955,0.279


## 2. Who changes: home or away team?

`card_diff` and `foul_diff` are differences. Modelling each team's count separately shows which side moves.

In [3]:
sides = pd.DataFrame([fit_ghost_model(analysis, outcome) for outcome in ["HY", "AY", "HF", "AF"]])
sides["meaning"] = ["home yellows", "away yellows", "home fouls", "away fouls"]
sides.round(3)

,outcome,coef,ci_low,ci_high,p_bootstrap,mde,full_crowd_mean,meaning
0,HY,-0.199,-0.352,-0.045,0.024,0.220,1.590,home yellows
1,AY,-0.424,-0.581,-0.267,0.000,0.224,1.751,away yellows
2,HF,1.067,0.632,1.502,0.000,0.622,10.259,home fouls
3,AF,0.053,-0.411,0.517,0.828,0.663,10.539,away fouls


## 3. How much of the card change do fouls account for?

A simple counterfactual on raw averages (no controls, so it is only an illustration):

- Take the **cards-per-foul rates from full-crowd matches** (home and away separately: total yellows ÷ total fouls).
- Apply them to the **foul counts observed in ghost matches**.
- The result is the card difference we would expect in ghost matches if only fouls had changed and the referees' card decisions had not.

In [4]:
full = analysis[(analysis["crowd"] == "full") & (analysis["low_fouls"] == 0)]
ghost = analysis[(analysis["crowd"] == "ghost") & (analysis["low_fouls"] == 0)]

# Cards per foul with fans, computed as total cards / total fouls (more stable than averaging match ratios)
home_rate_full = full["HY"].sum() / full["HF"].sum()
away_rate_full = full["AY"].sum() / full["AF"].sum()
print(f"Full crowd: home {home_rate_full:.3f} and away {away_rate_full:.3f} yellows per foul")

actual_full = full["card_diff"].mean()
actual_ghost = ghost["card_diff"].mean()
fouls_only_ghost = (away_rate_full * ghost["AF"] - home_rate_full * ghost["HF"]).mean()

total_change = actual_ghost - actual_full
fouls_part = fouls_only_ghost - actual_full
rate_part = actual_ghost - fouls_only_ghost

decomposition = pd.DataFrame({
    "card_diff": [actual_full, fouls_only_ghost, actual_ghost],
}, index=["full crowd (actual)", "ghost, if only fouls had changed", "ghost (actual)"])
print(decomposition.round(3))
print()
print(f"Total raw change:                 {total_change:+.3f}")
print(f"  from the change in fouls:       {fouls_part:+.3f}  ({fouls_part / total_change:.0%})")
print(f"  from the change in cards/foul:  {rate_part:+.3f}  ({rate_part / total_change:.0%})")

Full crowd: home 0.155 and away 0.166 yellows per foul
                                  card_diff
full crowd (actual)                   0.160
ghost, if only fouls had changed      0.046
ghost (actual)                       -0.023

Total raw change:                 -0.183
  from the change in fouls:       -0.114  (62%)
  from the change in cards/foul:  -0.069  (38%)


In [5]:
# For reference: cards-per-foul rates in ghost matches
home_rate_ghost = ghost["HY"].sum() / ghost["HF"].sum()
away_rate_ghost = ghost["AY"].sum() / ghost["AF"].sum()
pd.DataFrame({
    "full crowd": [home_rate_full, away_rate_full],
    "ghost": [home_rate_ghost, away_rate_ghost],
}, index=["home yellows per foul", "away yellows per foul"]).round(3)

,full crowd,ghost
home yellows per foul,0.155,0.127
away yellows per foul,0.166,0.131
